$$\require{mchem}$$
# **Analysis of MD Simulations** 

<span style="color:darkred"> **Summary** :  The analysis of an MD simulation should answer important questions: 
* <span style="color:darkred">Does the model system evolve dynamically at equilibrium conditions?
* <span style="color:darkred">Is the simulation  reasonably converged?
* <span style="color:darkred">What is the global or  local flexibility of the model?
* <span style="color:darkred">Which are the most likely conformations?
* <span style="color:darkred">In the case of comparing various MD models for the same biomolecule, which one is the most stable?
<span style="color:darkred"> Answers to  these questions are provided by  monitoring the  evolution of appropriate molecular *descriptors* and/or by performing statistical analysis of their values. In this session, we present various descriptors and analysis tools that are particularly adequate for analyzing  aptamer simulations.   

## **Structural and energetic descriptors** 

### **MD analysis: more software and scripting** 
Once that the production phase of an MD simulation is completed, the  trajectory files must be processed using different programs  to distill the structural and energetic properties of interest. Thus, [CPPtraj](https://github.com/Amber-MD/cpptraj) and [MDanalysis](https://www.mdanalysis.org/) are widely used programs that implement a large variety of analysis methods. However, depending on the type of system being analyzed, other specialized programs (e.g., `x3dna-dssr`) are usually  employed  so that many users create scripts for customizing their   protocols. In this respect, the `APTAMD` scripts process in a semiautomatic manner the trajectory files from aptamer simulations using `CPPtraj` and  other programs. The default options and settings taken by these scripts  allow us to to extract and assess rapidly the key  results and implications of the MD simulations. However, further questions may probably arise that would prompt us to rerun analyses with different options or modify/create scripts, etc.      

### **Structural descriptors: from global to local views of biomolecules**   
Due to the high dimensionality of the conformational space,   a structural analysis focuses always on a certain selection of degrees of freedom (DOF).  For example,   the  torsional angles among backbone atoms in aptamer molecules would be particularly interesting as they determine the overall conformation of the nucleic acid chain.   Although  possible,  a direct comparison between two or more MD structures in terms of  torsion angles would be extremely cumbersome. Therefore, a drastic reduction  in the number DOFs is done by defining **global descriptors**, which  are usually averages of coordinate differences, or by selecting a few internal coordinates that depict local details (e.g., H-bond distances involved in specific base pairing contacts). Using  MD analysis programs,  many structural descriptors can be calculated and  all kind of internal coordinates can be measured. Thus, in what follows, we only present the descriptors that are readily calculated with the APTAMD scripts.        

#### **RMSD** 
A simple and widely used measure of the difference between two structures, $I$ and $J$, is the root mean squared deviation [RMSD](https://userguide.mdanalysis.org/stable/examples/analysis/alignment_and_rms/rmsd.html) among the  coordinates of $N$ atoms ${\bf{x}}_i$,

$$ RMSD_{(I,J)} = \sqrt { \frac {  \sum\limits_i^N w_i \left( {\bf{x}}_i^I - {\bf{x}}_i^{J}  \right)^2 } { \sum\limits_i^N w_i } }  $$

where $w_i$ is the weight assigned to atom $i$. 

* **Q1** Which weight values $w_i$ can be used in RMSD calculations?
* **Q2** Can $ RMSD_{(I,J)} $  be considered a *distance* between $I$ and $J$? 

RMSD are not meaningful  unless the coordinate sets ${\bf{x}}^I$ and ${\bf{x}}^J$ are oriented with respect to each other.  For this reason, one set of coordinates is taken as the *reference structure* $J$ and the problem is to find the best transformation that *superimposes* the other coordinate $I$ set upon $J$. The most used RMSD algorithm 
works by initially translating the coordinate set of $I$ so that it has the same center of mass (COM) of $J$,

$$ {\bf{x}}_i^I\prime = {\bf{x}}_i^I - {\bf{x}}_{COM}^J$$   

The translated structure $I$ is then reoriented by applying a rotation matrix $\bf{U}$  that operates upon all the ${\bf{x}_i^I}\prime$ vectors.  In this way, the RMSD value becomes a function of $\bf{U}$ :

$$ RMSD_{(I,J)}\left( {\bf{U}} \right) = \sqrt { \frac {  \sum\limits_i^N w_i \left( {\bf{U}\bf{x}}_i^I\prime - {\bf{x}}_i^{J}  \right)^2 } { \sum\limits_i^N w_i } }  $$

The $U$ that produces the smallest deviation is found by *minimizing* the function $ RMSD_{(I,J)}\left( {\bf{U}} \right)$ , what is a problem that admits exact solution  when rotations are represented by  [*quaternions*](https://arxiv.org/pdf/physics/0506177) (not discussed here). 

From the description of the RMSD formulas, it is clear that some choices must be made to perform a  meaningful RMSD analysis:       

* **Atom selection** Not all coordinates  in biomolecules   are included in an RMSD analysis. For example, to describe large-scale movements, the atoms to be considered are usually restricted only to the backbone atoms. In practical terms, users need to specify the **atomic mask** defining the selected atoms.
* **Sieve of MD frames** Since RMSD calculations have a non-negligible cost and the overall geometry in MD frames separated by short times ( 1-10 ps) tends to be correlated, the ensemble of MD frames is sieved to perform RMSD  only for every given number of frames.
* **1D RMSD** The RMSD between **one reference structure** and a set of $M$ MD frames measures the similarity of the MD ensemble to the reference. It can provide insight into the overall movement and flexibility of the system. However, it must be noticed that different structures can have identical or nearly-identical RMSD from the same reference.
* **Reference selection** Users need to select a proper reference.  There is no requirement that the reference structure belong to the trajectory or even be of the same biomolecular species. What is really indispensable is  to ensure a one-to-one correspondence between atoms in the reference and atoms in the MD frames.   
* **2D RMSD**  Calculating the RMSD of each frame in a trajectory to all other frames yields much more information. A 2D RMSD matrix is then obtained that is conveniently analyzed using graphical or statistical tools. However, the cost of a 2D RMSD increases with $\mathcal{O}\left( M^2 \right)$ and demands considerable memory and disk space! 2D RMSD calculations can be performed among MD frames from the same or distinct simulations

The APTAMD scripts utilize the `CPPtraj` program to carry out RMSD analyses, which can be fine-tuned using multiple [options](https://amberhub.chpc.utah.edu/rmsd/). Using ChimeraX, structure superposition and RMSD calculations are common operations performed with the `match_maker` tool.


[x]:![](PNG/super.png)
<img src="./PNG/super.png" width="980" height="640" style="display: block; margin: 0 auto">

**Fig 1** Superposition of a truncated aptamer structure ($I$) onto a full-length aptamer ($J$). Note the atom mask syntax employed to specify the target atoms for the RMSD superposition.  

* **Q3** Consider the superposition shown in **Fig 1**. How similar or dissimilar is the truncated aptamer $I$ with respect to the full-length structure $J$?   

#### **Radius of gyration $R_{gyr}$**

The radius of gyration $R_{gyr}$ is a measurement of the distribution of $N$ atoms in a molecular structure $I$ with respect to its center of mass: 

$$ R_{gyr,I}  = \sqrt { \frac {  \sum\limits_i^N w_i \left( {\bf{x}}_i^I - {\bf{x_{COM}}}^I  \right)^2 } { \sum\limits_i^N w_i } }  $$

where $w_i$ is the atomic mass of atom $i$. A low $R_{gyr}$ value is indicative of a compact shape, in which the atoms in a structure are close to their COM, whereas and a high radius $R_{gyr}$ is more indicative of an elongated shape. Likewise the RMSD calculations, $R_{gyr}$ is usually obtained for some atomic selection (e.g, backbone atoms) and calculated using `CPPtraj` over a subset of  MD frames with $M$ elements. Inspection of the evolution of $R_{gyr}$ along a MD trajectory may signal sharp transitions.  

#### **Interaction network fidelity (INF) index**

The so-called interaction network fidelity index [INF](https://rnajournal.cshlp.org/content/15/10/1875)  results from  operations involving a set of characteristic interactions in a reference structure ($S_{ref}$) and equivalent sets in  MD frames ($S_I$).  The sets required to evaluate the INF index are:  

1.  The set of common interactions between the two structures, which are counted as true positives ${T_P} =  {S_{ref}} \cap {S_I} $  (set intersection).
2.  The set of the interactions in the reference structure that are *not* present in the MD snapshot, which are counted as false positives, ${F_P} = {S_I}\backslash {S_{ref}} $ (set difference).
3. The set of  interactions absent in the MD snapshot but present in the reference structure, which are counted as false negatives ${F_N} =  {S_{ref}}\backslash {S_I}  $.

The INF index is calculated as the *Mathews correlation coefficient* considering true and false positives and negatives: 
$$INF = \sqrt {\left( {\frac{{\left| {{T_P}} \right|}}{{\left| {{T_P}} \right| + \left| {{F_P}} \right|}}} \right)\left( {
\frac{{\left| {{T_P}} \right|}}{{\left| {{T_P}} \right| + \left| {{F_N}} \right|}}} \right)} $$

where $\lvert \:\:\: \lvert$ represents the set *cardinality* (i.e., number of elements).  If the given MD  snapshot reproduces all the interactions of the reference structure, then $\lvert F_P \lvert = \lvert F_N \lvert=0 $, and $INF=1$. When the MD snapshot does not reproduce any of the interactions of the reference structure, then $INF=0$ because $\lvert T_P \lvert=0$. 

To determine the characteristic interactions in aptamer molecules, the best alternative is to employ the `x3dna-dssr` program as discussed in Session 2. From the atomic coordinates of a given nucleic acid model, `x3dna-dssr`  identifies both base pair interactions and non-pair interactions (i.e., base stacking or base contacts) and categorizes them using well-defined standard protocols. Note, however, that for the sake of INF calculations, a base-pairing contact is only annotated in terms of the pair of nucleotide IDs regardless of the type of base pairing interaction. 

The INF index can be considered as a global structural descriptor related to the 2D (secondary) structure of  aptamers.  Hence, INF complements well the 3D information captured by other descriptors such as  RMSD and $R_{gyr}$. In this way, the INF, RMSD and $R_{gyr}$ analyses can reveal significant information about the dynamic behavior along MD simulations. For example, we see in **Fig 2** how the backbone conformation of  an aptamer molecule departs from the starting point (as shown by the RMSD plot), adopts a more compact form ($R_{gyr}$ tends to decrease) and undergoes changes in secondary structure (only 50% of initial contacts are preserved). 

[x]:![](PNG/rmsd_inf_plot.png)
<img src="./PNG/rmsd_inf_plot.png" width="720" height="480" style="display: block; margin: 0 auto">

**Fig 2** . Evolution of INF, RMSD and $R_{gyr}$ as a function of the number of MD frames. INF and RMSD are calculated with reference to the initial model considered in the simulation.  The mean and standard deviation of the data sets are also indicated. 

* **Q4** How would you convert the plots in **Fig 2** into *time* evolution plots?

* **Q5** Discuss whether or not the **Fig 2** plots are compatible with a well *equilibrated* MD trajectory.
 


Using the  `do_struct.sh` script in APTAMD with the `DO_INF=YES` option , the INF calculations are readily performed as the script extracts the MD coordinates, parses the output files from `x3dna-dssr` and performs descriptive statistics. Besides plotting the INF evolution during the simulation, the identity and abundance of specific base contacts are written in a text file named `BP_NP_stat.dat`. 

```
=============================== 
    BASE PAIRING CONTACTS 
=============================== 
   IRES    JRES   ABUND.(percen) 
================================ 
 DC_14 ...  DG_43   100.0000 
  DC_7 ...  DG_51    99.9995 
 DG_13 ...  DC_44    99.9995 
 DA_15 ...  DT_42    99.9973 
 ...
=============================== 
  NON PAIRING (FULL) CONTACTS 
=============================== 
   IRES    JRES   ABUND.(percen) 
================================ 
 DA_15 ...  DG_16    99.9880 
  DG_9 ...  DA_10    99.9430
...
```

#### **Molecular surface** 

The `CPPtraj` program performs fast calculations of molecular surfaces. Hence, the resulting area $A$ values  are also structural descriptors that have either global or local character depending on the *mask* of solute atoms being selected. For example, one may be interested in measuring the solvent-exposed area of the whole aptamer or that of a binding site enclosed by a set of residues. 

Different types of molecular surfaces can be defined, the **solvent-accesible surface** (SAS) being the most popular one in molecular modeling. To calculate a SAS, both atomic coordinates ${\bf{x}}_i$ and *radii* $r_i$ are needed together with a *probe radius* $r_p$. Atomic radii for each *atom type* are assigned automatically during the molecular edition stage and the probe radius is normally taken as $r_p=1.4 \:\mathring(A)$ mimicking a water molecule . Hence, solute atoms are represented as overlapping spheres with radii $r_i$ located at  each ${\bf{x}}_i$  while  solvent is a rigid sphere with radius $r_p$ that rolls over the atomic spheres. The SAS is then  defined by the points visited by the center of the  probe sphere and the total surface area is calculated by mesh triangulation.

[x]:![](PNG/surf.png)
<img src="./PNG/surf.png" width="720" height="480" style="display: block; margin: 0 auto">

**Fig 3**. View of an aptamer SAS generated by ChimeraX. The inset shows schematically the calculation of SAS. Evolution of SAS along a MD simulation is also shown.   

SAS calculations are activated by the `DO_SURF=YES` option in `do_struct.sh`.  

* **Q6** Using ChimeraX, fetch the PDB biounit with code 1EN9. Use the `surf` and `trans 50 target s` commands to represent a translucent SAS around the model. Can you visualize the major and minor *grooves* of the DNA double helix ?

   

### **Energy scoring of MD simulations: *approximate* methods are needed**

Clearly, the energetic analysis of an MD simulation may complement  the assessments made with the various structural descriptors. However, although conventional MD simulations in explicit solvent evolve both the coordinates and the MM energy components, they cannot  provide  rigorous estimates of the *absolute* free energy $G\left(N,P,T\right)$ of biomolecules in solution. 
It is true that there are MD-based methodologies that can rigorously calculate free energy *differences* connecting two thermodynamic states (e.g., $\Delta G_{binding}$). However, they are computationally expensive, quite difficult to use and their study is beyond the scope of this tutorial. Moreover, we need to evaluate a $G$ descriptor of conventional MD simulations rather than $\Delta G$ differences.  Instead we  resort to an approximated method, the so-called Molecular Mechanics/Poisson−Boltzmann Surface Area (**MM/PBSA**) protocol, which   directly *estimates* the free energy of a solute molecule by combining its molecular mechanics (MM) energy with its solvation free energy. 

The [MM-PBSA](https://pubs.acs.org/doi/abs/10.1021/acs.jcim.8b00805) method is better introduced as a **physically-based scoring function**. Thus, the conventional MM-PBSA energy $G$ of the solute atoms is computed as:

$$ G = E_{MM} + \Delta G_{solv}^{PB} +  \Delta G_{solv} ^{np} $$

where  $E_{MM}$ is the molecular mechanics energy , $\Delta G_{solv}^{PB}$  is the electrostatic solvation energy obtained from Poisson-Boltzmann calculations,and $\Delta G_{solv}^{np}$  is the non-polar part of solvation energy due to cavity formation and dispersion interactions between the solute and the solvent molecules. Typically, the MM/PBSA calculations are performed over a set of MD frames after having removed the coordinates of waters and ions. The evolution of the $G$ values with the amount of sampling serves us to assess whether the simulation is energetically equilibrated while the average value $\braket{G}$ would be the (approximated) free energy scoring of the whole or a part of the simulation.

#### **The PBSA implicit solvent model**
In the MM/PBSA expression, an effective **solvent continuum model** replaces the explicit description of water and counterions.  The *implicit* solvent is just a medium with a high dielectric constant $\epsilon_{out}=80$. The boundary between solute and solvent is determined by the solvent-accessible surface so that SAS calculations are needed. The interior of the molecule is assigned a low dielectric constant $\epsilon_ {in}=1,2, 4, ...$. At a first glance, $\epsilon_ {in}$ should be unity since the model keeps the solute atoms. However,  polarization effects not considered explicitly by  MM  methods  may cause a higher $\epsilon_ {in}$  to be more appropriate. As an additional approximation, the electrostatic and non-polar solvent effects are treated separately:
* The electrostatic part is computed by solving numerically the [Poisson-Boltzmann equation](https://apbs.readthedocs.io/en/latest/background.html) to obtain  the *total* electrostatic potential $\Phi$ over the grid points $\bf{x}$ of a 3D mesh. $\Phi$ is created by the solute atoms and the reaction field generated by the surrounding dielectric media.
    $$\nabla \cdot \left[ \epsilon(\mathbf{x}) \nabla \Phi(\mathbf{x}) \right] - \bar{\kappa}^2(\mathbf{x}) \sinh\left[\Phi(\mathbf{x})\right] = -4\pi \rho(\mathbf{x})$$
where $\kappa$ is the inverse Debye length parameter, which depends on the ionic strength ans is null inside the solute, and $\rho$ is the  charge density due to  the solute atoms.  Note that the dielectric constant $\epsilon$ is a function of the position of the grid points $\bf{x}$. From the difference between $\Phi(\epsilon_{out}=80)$ and $\Phi(\epsilon_{out}=\epsilon_{in})$, the value of $\Delta G_{solv}^{PB}$ is readily obtained.
* The non-polar contribution $\Delta G_{solv}^{np}$ is estimated by *empirical* expressions that depend on the total SAS area and volume.

[x]:![](PNG/Grid.png)
<img src="./PNG/Grid.png" width="720" height="480" style="display: block; margin: 0 auto">

**Fig 4** Solving the PB equation yields the electrostatic potential over a 3D grid of points. Many technical details and settings influence the solving of the PB equation. Adapted from [ref](https://doi.org/10.4155/fmc.10.6).

#### **Utilizing MM/PBSA energies**

It must be emphasized that, compared with the explicit solvent models combined with long-range PME methodologies, the implicit PBSA  model gives a *crude* and non-specific representation of solvent effects. Moreover, the $E_{MM}$ and $\Delta G_{solv}$ terms are necessarily *unbalanced*, leading to a large statistical uncertainty of the average $G$ values. Nevertheless, the intrinsic errors of the MM/PBSA scoring function are not expected to affect the **monitoring of energetic relaxation along a MD simulation**  (see the example in **Fig 5**) and they may be largely canceled when **comparing the $G$ scorings of different MDs** trajectories (see **Fig 6**). As a matter of fact, many MM/PBSA variants are routinely applied in a broad range of biomolecular modeling applications and there exist several tools for streamlining this type of calculations on MD ensembles like [gmx mmpbsa](https://valdes-tresanco-ms.github.io/gmx_MMPBSA)).        

[x]:![](PNG/complex.png)
<img src="./PNG/complex.png" width="720" height="480" style="display: block; margin: 0 auto">

**Fig 5** Analysis of the MD simulation of an aptamer molecule in complex with a glycoprotein. The simulation was started from a initial structure  obtained by molecular docking. 


* **Q7** On the basis of data shown in **Fig 5**, is the model  properly *relaxed* during the simulation? Is the MD simulation  well *equilibrated* ?

  

`do_mmpbsa.sh` is our script for carrying out MM/PBSA analysis.  This is a rather complex tool that produces energies using  the *conventional* MM/PBSA recipe as well as using other variants (not discussed here). It works with a large number of programs (e.g, `CPPtraj`, `parmed`, `sander`, `pbsa` from the Amber package , but also  many auxiliary scripts and programs).  Of particular interest maybe the ability of `do_mmpbsa.sh` to include a subset of strongly-interacting $\ce{Na+}$ ions plus hydrating waters, which are then treated as solute atoms. This option, which is not provided by other MM/PBSA driver tools,  can be particularly useful to analyze the MD trajectories of the highly-charged  aptamer systems because implicit  solvent models may lead to extreme overestimation of electrostatic repulsions among phosphate groups. Similarly,  an inner dielectric constant of $\epsilon_{in}\: \sim \: 2-4$ may be also recommendable. 

[x]:![](PNG/mmpbsa.png)
<img src="./PNG/mmpbsa.png" width="720" height="480" style="display: block; margin: 0 auto">

**Fig 6** Comparison of $G$ plots derived from two MD models of the same aptamer sequence. The mean values of $G$ (in kcal/mol),   standard error (SE) and limiting block-averaged error (LBE) are indicated. The MM/PBSA calculations were performed with and without the inclusion of explicit $\ce{[Na(H2O)6]+}$ complexes nearby the phosphate groups. The number of $\ce{[Na(H2O)6]+}$ complexes is determined by previous statistical analysis of the closest $\ce{Na+}\cdot\cdot\cdot$aptamer distances (see the inset histogram). The presence of the   $\ce{[Na(H2O)6]+}$  complexes affects the magnitude of the $G$ fluctuations. 

* **Q8** According to the mean energies reported in **Fig 6**, which is the most likely model? What is the $\Delta G$ between the two models? Do you think that MM/PBSA overestimates or underestimates the $\Delta G$  values?
* **Q9** What MM/PBSA settings could be investigated to further support relative stability predictions as that in **Fig6**?  

#### **MM/PBSA scorings of MDs may have a large uncertainty**

Statistical analysis of the MM/PBSA energies are also performed by `do_mmpbsa.sh` to find the corresponding mean values and their uncertainty. Since it may be convenient to use only a fraction of the accumulated data, a segment of the MD trajectory can be selected for statistics. In any case, one should expect a significant uncertainty of the mean values due to the limited sampling and the approximated character of the MM/PBSA method. Thus, the instantaneous values of the MM/PBSA energy components and relative differences oscillate rapidly on the short timescales (<$\rm{ns}$)  and exhibit lower-amplitude oscillations on longer time scales (see for example **Fig 5-6**). This means that the common standard error (SE) will underestimate the uncertainty in a series of $G$ values  because such  data  will contain a certain *long time correlation*. Therefore, the statistical uncertainty of the average values is also assessed by computing the **block-averaged standard errors of the mean** (BE), which helps eliminate self-correlation effects in time-series of data.    

[x]:![](PNG/bse.png)
<img src="./PNG/bse.png" width="480" height="320" style="display: block; margin: 0 auto">

**Fig 7** Example of LBE determination for a sample of 1000 $G$ MMPBSA energies.

Using the BE procedure , the MD trajectories are divided into segments (*blocks*) with a block size $M_{block}$ that ranges from a minimum up to a maximum equal to $\sim \frac{1}{2}, \frac{1}{4}, \frac{1}{8}, ...$ of the total number of frames. The BE value is obtained by computing first the mean value for each block, secondly, the mean of the block means, and, finally, the standard error for the latter. The limiting value of BE versus block size can be taken as an upper limit to the statistical uncertainty. There is no unique BE procedure since the limiting errors depend on the choice of $M_{block}$. In the APTAMD tools, we select $\max{M_{block}}$ equal  to a quarter of the total number of frames. Other [BE algorithms](https://manual.gromacs.org/current/onlinehelp/gmx-analyze.html) have been proposed that  employ a  non-linear fitting equation to determine the LBE values, but they are more prone to numerical errors and instability.        

```
# Example of a G_MMPBSA.med file generated by do_mmpbsa.sh   
# PREFIX G_MMPBSA(CMPLX) USING ALL DATA NDAT= 999
 Mean       -5521.072  
 Max        -5447.862  
 Min        -5594.753  
 SE             0.870  
 LBE            2.367  
# SE: standard error 
#LBE: limiting block error estimate
```


## **Clustering Analysis**

Clearly, it is desirable to be able to select from large MD ensembles a smaller, *representative*, set of conformations. This is even more important in the case of aptamer simulations because such  *representative* structures condense the structural and flexibility features of the *in silico*  aptamer models.       

In general, a cluster analysis groups together *similar objects*, from which the representatives can be extracted. There is no universal clustering method and a large number of algorithms have been developed. All of them  require a measure of the *distance* between pairs of objects, $\left( I,J \right)$. When comparing conformations of biomolecules, the RMSD is the obvious distance metric to use. By construction, RMSD is an Euclidean distance but other metrics can be used in conformational clustering based on torsional angles (e.g, a *Manhattan* distance between torsion angles  $d_{(I,J)}= \sum\limits_{m=1}^{N_{tor}} \lvert \omega_m^I - \omega_m^J \lvert $). Nonetheless, **RMSD-based clustering methods are preferred** to characterize the overall conformation of biomolecules. 

### **Hierarchical linkage clustering: A favorite method in molecular modeling**

Most of the clustering algorithms implemented in MD analysis software such as [CPPtraj](https://amberhub.chpc.utah.edu/clustering-a-protein-trajectory/) can be classified as **k-means partitioning** or  **hierarchical linkage**  methods. Here, we discuss only the linkage methods because they are used in the APTAMD protocol.  These methods perform the following sequence of actions through which clusters are formed and merged:

1. Compute the **2D RMSD distance matrix** $\bf{D}$ among the selected set of MD frames containing $M$ elements.
2. Start with $ N_{clus} = M $, that is, as many clusters as frames. Set the initial distance matrix between clusters ${\bf{D}}_{clus}=\bf{D}$  
3. Loop until the distance between the *closest* pair of clusters exceeds a predetermined value, $D_{clus}\left(O,P\right) \geq RMSD_{thres}$
    1. Find the *closest* pair of clusters $\left(O,P\right)$ that has the lowest $D_{clus}$ value.
    2. Merge  $\left(O,P\right)$ into a single cluster
    3. Update $N_{clus}$ and ${\bf{D}}_{clus}$
4. Select a representative structure for each cluster $O$ as the conformation $I$ that is closest to the average structure of the cluster.

As shown in **Fig 8**,  hierarchical linkage clustering can be represented visually by constructing a *dendrogram*, which indicates the relationship between the conformations in the data set.

[x]:![](PNG/linkage.gif)
<img src="./PNG/linkage.gif" width="640" height="320" style="display: block; margin: 0 auto">

**Fig 8** Animated GIF showing the basic steps of a  hierarchical linkage algorithm until just a single cluster remains(see also the pseudo-code cell below).  The dendogram at right represents the $M$ structures along  the $x$ axis while the  y axis indicates the intercluster distance.

* **Q10** What is the computational cost of linkage clustering?
* **Q11** Does linkage clustering depend on the ordering of the data set? 

In [ ]:
# Pseudo-code representation of linkage clustering
# --- 1. Cluster settings ---
k =  10         # Maximum number of clusters
threshold = 4.0 # Maximum RMSD distance between clusters  
# --- 2. Compute all pairwise RMSD distances ---
M = len(conformations)
D = np.zeros((M, M))
for I in range(M):
    for J in range(I+1, M):
        D[I,J] = D[J,I] = rmsd(conformations[i], conformations[j])
# --- 3. Initialize: each conformation is its own cluster ---
clusters = [{I} for I in range(M)]
Dclus= D 
# --- 4. Agglomerative loop ---
while len(clusters) > k or min_interdist(clusters, Dclus) <= threshold:
    # Find the closest pair of clusters
    O, P = argmin ( Dclus )
    # Merge and update distance matrix
    clusters[O] = np.concatenate ( clusters[O] , clusters[P] )
    del clusters[P]
    Dclus = update_distances (clusters, D)  #  Dclus[O,P] =  average of RMSD(I,J) for every I in cluster O and J in cluster P  
# --- 5 Select cluster representatives
for O in range(len(clusters)):
    representative[O] = mediod( clusters[O], D ) 


### **Guidelines for performing cluster analysis**

Understanding the basics of the linkage algorithm is required to perform a meaningful clustering analysis. Thus, a careful choice of input settings is necessary to produce clusters in a timely manner. Moreover, it is normal to try different settings to optimize the cluster output  as well as to answer specific questions. 

* **Atom selection** Only backbone atoms are usually  considered in the computation of the RMSD distance matrix. For aptamers, a first atom selection may comprise the whole chain. For long sequences,  it may be acceptable to exclude the first  and  last $\sim$ 3-5 nucleotides  since the 3'- and 5'-ends are normally highly flexible.  Depending on the outcome of the clustering,  the atom selection can be refined to include only the residues that adopt more structured conformations. In this way, clustering analysis will discriminate more clearly between flexible and rigid domains and reveal the nature of interdomain motions.     

* **Sieve of MD frames**  The computational  effort needed to build and manipulate the RMSD distance matrix limits severely the number of frames $M$. For a medium-sized aptamer with 30-40 nucleotides, it is not recommended to process more than $M \sim 10000-20000$ frames. The sieve parameter should be adjusted accordingly, depending also on the length of the simulation.  If necessary,  the  excluded MD frames  can be mapped rapidly into the resulting clusters.    

* **RMSD threshold** It is difficult to know *a priori* which RMSD threshold should be applied. Therefore, it is probably wise to run batches of clustering calculations varying the threshold parameter (e.g., 2.0, 3.0, 4.0 and 5.0 $\rm{\mathring{A}})$.

The APTAMD script `do_cluster.sh` drives clustering analysis using the `CPPtraj` program. As usual, it organizes all the input/output data files and assigns reasonable (initial) values to the clustering options. For example, the RMSD distance metric between MD frames is calculated using the coordinates of backbone heavy atoms.  A variable `EPS` can be declared in the input file of `do_cluster.sh` listing several RMSD threshold values (e.g., `EPS="3.0 4.0 5.0"`).  

Of course, the output information should be examined in detail to assess the actual significance of the cluster representatives and to compare the results of various simulations. Using the hierarchical linkage approach, particular attention should be paid to:   

* **Number of clusters**  $N_{clus}$ is very sensitive to the value of the RMSD threshold parameter and the dynamic properties of the system. According to computational experience, a moderate value of $N_{clus}$ around 10-30 may be optimal for further analysis and visualization.   
* **Cluster population** It is desirable that the MD frames are more or less evenly distributed among a few top clusters. The two limiting situations, (1) one cluster concentrating the large majority  of the MD frames and (2) a high number of clusters having all low populations, should be avoided as they give little or none insight. 
* **Visualization of representatives**  Using Chimerax or other software, the most populated cluster representatives can be superposed and visually inspected. Take note of the similarities and differences among the models. Interrogate the models to see if they can explain the trends revealed by the structural and energetic descriptors.

```
# Summary of cluster analysis generated by CPPtraj
#Cluster   Frames     Frac  AvgDist    Stdev Centroid AvgCDist
       0     4704    0.379    2.023    0.513     5518    3.532
       1     2613    0.211    2.069    0.512    11477    4.372
       2     2269    0.183    1.966    0.489     1180    3.690
       3      914    0.074    2.086    0.482     8978    3.521
       4      214    0.017    2.223    0.494     3627    3.925
       5      195    0.016    1.771    0.595     2538    3.773
```

**How can we assess the performance of a clustering analysis?** In principle, this question might be answered by computing one or more statistical measurements of cluster efficiency (e.g., [Silhouette coefficient](https://en.wikipedia.org/wiki/Silhouette_(clustering))). However, in complex scenarios involving several  MD simulations of aptamer molecules, it may be more adequate to consider two or clustering settings and apply chemical intuition on a case-to-case basis.  

###  **An example of aptamer clustering**    

To exemplify the convenience of testing various settings,  **Fig 9** summarizes the results of the clustering analysis of a particular MD simulation. Clustering on the coordinates of the whole backbone chain (left) demands a high RMSD threshold of $5.0\rm{\mathring{A}}$ to yield significantly-populated clusters. Upon close examination, it is concluded that the hairpin motif has a similar folding in the top  representatives. This is confirmed  by a second clustering using only the backbone atoms of the  first 20 nucleotides,  a lower threshold  $2.5\rm{\mathring{A}}$ groups the conformations into a few highly populated clusters. The second half of the chain (:21-40) exhibits some structural similarity albeit with larger flexibility as indicated by the results of the third clustering analysis (right). Therefore, this aptamer model possesses two nearly-independent domains that undergo ample fluctuations in their relative orientation. 

[x]:![](PNG/clust_1.png)
<img src="./PNG/clust_1.png" width="1024" height="640" style="display: block; margin: 0 auto">

**Fig 9**  Three different clustering settings applied to a 2.5 $\rm{\micro s}$ MD simulation of a 40-mer aptamer.  ChimeraX views correspond to superposed models using the :1-20, :1-20 and :21-40  backbone atoms (from left to right).  

* **Q12** Besides visualizing them, how can we use the cluster representatives to gain further structural insight?

* **Q13** On the basis of data in **Fig 9**, which **truncation strategy** would you propose  to remove non-essential nucleotides from the original sequence ?

* **Q14** The Figure below  shows the results of a preliminary clustering analysis of a 5.0 $\rm{\micro s}$ MD simulation of a 57-mer aptamer. Which new settings would you consider to repeat the clustering calculations?

[x]:![](PNG/clust_2.png)
<img src="./PNG/clust_2.png" width="720" height="540" style="display: block; margin: 0 auto">

## **Amount of sampling and convergence of simulations**

### **Entropy measures the size of phase space** 
Clearly, the *amount*  of conformational sampling is a major factor determining the reliability of long-time MD simulations of flexible aptamers. Of course, clustering analysis and RMSD plots provide useful information about  conformational sampling, but they do not give  objective measures of the amount of sampling performed by the MD simulations. Alternatively, the *configurational entropy* $S_{conf}$ of a biomolecule, as defined in Statistical Mechanics, constitutes a probabilistic measure of the size of the phase space that is *accessible* to the system of interest:

$$S_{conf} = -k_B \int \rho \left({\bf{R}}\right)  \ln \left[ { \rho\left({\bf{R}}\right) } \right] d{\bf{R}} $$

where ${\bf{R}}$ is the array containing the molecular coordinates (the multidimensional phase space encompasses all the ${\bf{R}}$ points).  In the entropy formula, $ \rho \left({\bf{R}}\right)$ is the probability density function (PDF) that determines the probability that the molecule adopts a given configuration (e.g.,  $ \rho \left({\bf{R}}\right)d{\bf{R}} \equiv$ probability). Thus, $S_{conf}$  turns out to be an statistical descriptor associated to the PDF, more specifically, to the effective spread of  $ \rho \left({\bf{R}}\right)d{\bf{R}}$ all over the phase space.   

### **Measuring the size of MD sampling: conformational entropy calculations**   
The ultimate goal of any MD simulation is to provide enough data to reconstruct the *exact* PDF associated to the corresponding phase space.  As a matter of fact, knowing the *exact*  $ \rho \left({\bf{R}}\right)d{\bf{R}} $ would allow us to calculate all properties of interest. Unfortunately,  such a goal is not generally accessible excepting for small molecular systems with just a few degrees of freedom. However,  any  estimation of $S_{conf}$ retrieved from an MD ensemble would yield a valuable  size measurement. This is still a challenging problem and approximations are thereby necessary.    

To simplify the problem of configurational entropy calculations, we  pursue the following strategy:
1. Selection of a subset of  coordinates. The natural choice is the set of torsion angles $\{ \phi_i \}_{i=1,N_{tor}}$ around rotatable bonds that describe  conformational motions.
2. *Discretization* of the phase space defined by the continuous torsion angles. Every  $\{ \phi_i \}_{i=1,N_{tor}}$ array is then  transformed into a set of integer numbers $\{ a_i \}_{i=1,N_{tor}}$ with $a_i \in (1,2,3)$  labeling the discrete  states of each torsion. 
3. Build up approximations to the **probability mass function** (PMF) $P_{conform}\left(a_i\right)$ to calculate the **conformational entropy** $S_{conform}$. To fulfill this task, it is possible to devise efficient methods basing on the mutual information expansion of the full PMF (not discussed here).

[x]:![](PNG/sconform.png)
<img src="./PNG/sconform.png" width="720" height="540" style="display: block; margin: 0 auto">

**Fig 10** Workflow for $S_{conf}$ calculations. The torsion angles $\{\phi_i^k\}$ of all $i$-rotatable bonds are calculated at each MD $k$-frame using CPPtraj. Histrogramming of the $\{\phi_i^k\}$ sets gives univariate PDFs whose peaks and minima define the boundaries between the discrete conformational states. Discretization $\{\phi_i^k\}\longrightarrow \{a_i^k\}$ gives a column array for each torsion. Pasting all $\{a_i\}$ vectors gives a matrix whose rows define uniquely the conformational state of each MD frame. From this matrix, it is possible to build up statistically robust estimations of PMFs for every torsion, pairs of torsion, triads, ...which ultimately lead to the $S_{conform}$  values.     

The algorithms sketched in **Fig 10** are implemented in the [CENCALC program](https://github.com/dimassuarez/cencalc_quicksort). CENCALC transforms the time series containing the values of the rotatable dihedral angles into an array of integer numbers labeling the accessible conformational states and calculates a series of  multivariate PMFs $P\left(a_i\right), P\left(a_i,a_j\right), P\left(a_i,a_j,a_k\right), \: ... $ . The first-order entropy, $S_{conform}^{(1)}$ , is obtained as $\sum_i -P\left(a_i\right)\log{P\left(a_i\right)}$. CENCALC incorporates correlation effects among the dihedral angles within a predefined cutoff distance using advanced expansion techniques and removes entropy bias due to finite sampling  by shuffling the elements of the arrays of integer numbers prior to the entropy estimations. 

### **$S_{conform}$ with APTAMD**

Conformational entropy plots  ($S_{conform}$) for the aptamer simulations  can be calculated using the `do_sconform.sh` script, which  drives  all the tasks involved in the entropy calculations (e.g., dihedral angle selection and discretization, application of entropy methods, plotting and tabulating $S_{conform}$ values, etc.). Despite of the methodological intricacies of entropy methods, the default options taken by the CENCALC program are robust so that  `do_sconform.sh` is a user-friendly tool.       

In practical terms, it is useful to calculate $S_{conform}$ because:
* Plotting $S_{conform}$ as a function of the simulation time constitutes a clear-cut graphical assessment of the extent and convergence degree of conformational sampling.   
* $S_{conform}$  complements the  MM/PBSA energy scorings  with the limiting value of $-T S_{conform}$.

The utility of the entropy calculations is demonstrated in **Fig 11**. During the MD simulation, the hairpin loop populates several conformational states while the rest of the structure remains quite stable. The convergence entropy plot suggests that all the conformational transitions detected along the simulation were properly sampled. Note, however, that the quasi-convergence plateau of the $S_{conform}$ curve is achieved at the last stages of  the simulation.   

[x]:![](PNG/s_plot.png)
<img src="./PNG/s_plot.png" width="940" height="540" style="display: block; margin: 0 auto">

**Fig 11** Superposition of cluster representatives (right) obtained for a 2.5 $\rm{\micro s}$ simulation of a 27-mer peptide. Convergence entropy plot (center) and cutoff-entropy plot (right).  


## **Case study: Application of the MD analysis methods using the APTAMD scripts** 

### **Fast analysis of an MD simulation**

After completion of a conventional MD simulation driven by the `do_runmd` APTAMD script, it is time to start using the analysis scripts.

* Move to your project directory, which must contain the corresponding simulation directory,`apt_MD`. Examine the contents of  `apt_MD/5.PRODUCTION` directory and make sure that the MD is completed.
  
```bash
cd  project
cd  apt_MD/5.PRODUCTION
... 
```

* To speed up preliminary analysis tasks, create a script file within the `project/` directory with the name  `job_analysis.sh` or similar. Copy the code cell below into your script file and make the necessary changes!

In [ ]:
#!/bin/bash

# Script for running APTAMD analysis using mainly default options
if [ -z "$APTAMD" ]; then echo "APTAMD variable is not defined!" ; exit; fi
export NPROCS=8    # Maximum number of core processors to be used
# 
SYSTEM=""  # ----> Particularize here your system's name and MD type
MDTYPE=""

# Checkings
if [ -z ${SYSTEM} ]; then echo 'SYSTEM is not defined'; exit; fi
if [ -z ${MDTYPE} ]; then echo 'MDTYPE is not defined'; exit; fi
if [ ! -d ${SYSTEM}_${MDTYPE} ]; then echo "Simulation directory ${SYSTEM}_${MDTYPE} is not found!"; exit; fi

# Since only a basic usage of the scripts is needed, input_XXX.src files are not used!  

# Extract PDB files of snapshots + Analysis of Na+...aptamer contacts
$APTAMD/bin/do_snapshots ${SYSTEM}  ${MDTYPE}         

# Obtain structural descriptors (RMSD, RGYR, INF, SURF)
export DO_INF=YES     # Exporting variable declarations is another way of passing options to script  
export DO_SURF=YES
export SIEVE=10
$APTAMD/bin/do_struct ${SYSTEM}  ${MDTYPE}
unset SIEVE

# Clustering analysis: using different RMSD threshold parameters (EPS)
export SIEVE=50
export EPS="1.0 1.5 2.0 2.5"
$APTAMD/bin/do_cluster ${SYSTEM}  ${MDTYPE}
unset SIEVE

# MM/PBSA energy analysis using default options (Na+ not included)
$APTAMD/bin/do_mmpbsa ${SYSTEM}    ${MDTYPE}

# Conformational entropy calculations 
# including correlation effects within a given cutoff value
export CUTOFF="6.0 8.0 10.0 12.0 14.0 16.0"
$APTAMD/bin/do_sconform ${SYSTEM}  ${MDTYPE}  0  1

* Submit the `job_analysis.sh` task to the queue system.
  ```bash
  qapta job_analysis.sh
  ```
   The job will take about 30-60 min depending on the system size.

### **Know the analysis scripts**   
The script executes sequentially  the APTAMD scripts for MD analysis. Entering the script's name without arguments at the command line prints information about its function and options. Here, we provide only a brief summary:

1. `do_snapshots.sh` : extracts a subset of equally-spaced MD frames including coordinates of the closest waters and counterions. The coordinates of each frames are written in separate `.pdb` files suitable for visualization and further analysis. This script also performs a detailed analysis of $\ce{Na+}$···aptamer contacts. Output files are written by default in `6.ANALYSYS/SNAPSHOTS`. The PDB files are *gzipped*  (you can uncompress them with the `gzip -d ` command) while the `.png` files display a series of  histograms characterizing the  closest distances between  $\ce{Na+}$ and solute atoms.
   
2. `do_struct.sh`: calculates the structural descriptors. By default, RMSD, $R_{gyr}$ and INF are obtained using all the MD frames in the `5.PRODUCTION/md_*_solute.mdcrd` files. The most important script options to be considered are  `DO_INF`, `DO_SURF` and `SIEVE`.   Output files are saved in `6.ANALYSYS/STRUCT`. Plots are saved in `.png` files.

 3. `do_cluster.sh: ` performs clustering analysis. It is recommended to use  various RMSD thresholds specified as a list of values assigned to the `EPS` variable. Output files are located in  `6.ANALYSYS/CLUSTER/`. For every EPS value, a subdirectory is created  (e.g., `EPS_1.5`). The most relevant files are `summary.dat` and the `.pdb` files of cluster representatives named as `rep.c0.pdb, rep.c1.pdb, ...`.

4. `do_mmpbsa.sh`:  carries out the MM/PBSA calculations on the pdb files saved in   `6.ANALYSYS/SNAPSHOTS`. Hence, **a previous execution of `do_snapshots.sh` is a prerequisite to run   `do_mmpbsa.sh`**.  The output files are organized in the `6.ANALYSYS/MMPBSA/` directory. Note that a subdirectory is created for every value of the inner dielectric constant ($\epsilon_{in}$) employed  in the PBSA calculations (e.g. ,`PDIE_1`). **The most important files to be examined are `G_MMPBSA.med` and `G_MMPBSA.png`**. Do not consider  the rest of data files unless you know what you do!   

5. `do_sconform.sh`: computes various approximations to $S_{conform}$. By default, all the MD frames in the `5.PRODUCTION/md_*_solute.mdcrd` files are incorporated. Big aptamer molecules may require cutoff distances up to 20-30 $\rm{\mathring{A}}$ as defined in the `CUTOFF` variable. Output files are saved in `6.ANALYSYS/SCONFORM`. The essential information, that is, the convergence entropy plot, is summarized in the`.png` files.

   

### **Examining the output of the analysis scripts**

#### ***Hint*** As you progress in the examination of the results, it may be a good idea to incorporate the most important observations in this or other notebook:
* Insert new Markdown  or Raw cells.
* Copy text from Linux terminal to notebook cells.
* Small files can be also copied into the notebook directory and linked to the notebook file.
* Capturing image snapshots and pasting them in markdown cells is quite fast and effective.
etc.


#### `do_snapshots.sh`

* Move to `6.ANALYSIS\SNAPSHOTS` and perform the following actions:
  1. Uncompress the PDB files (`gzip -d *.pdb.gz`). 
  2. If you wish, visualize one or more `.pdb`file using `jmol`.
  3. Visualize the histogram files wit the command `eog hist_1*.png`. What is the probability $P$ that 1 $\ce{Na+}$ ion is at  < 3.0 $\rm{\mathring{A}}$ from aptamer atoms?   The same for 2 $\ce{Na+}$ ? 3 $\ce{Na+}$ ? ... 
  

#### `do_struct.sh`

* Move to `6.ANALYSIS\STRUCT` and perform the following actions:
  1. Inspect the  `options.txt` file. Which is the atomic mask for RMSD superposition? Which is the reference structure?
  2.  How can we modify the default mask and structural reference?      
  3. Visualize all the  `.png`files using `eog`. Describe in your own words the information conveyed by the structural descriptors. Is the system well *equilibrated* according to these plots?
  

#### `do_cluster.sh`

* Move to `6.ANALYSIS\CLUSTER` and perform the following actions:
  1. For each RMSD threshold value (`EPS_X` subdirectories), take note of the total number of clusters and the population of the top 3-5 clusters.
  2. Which is the most adequate RMSD threshold for your system?
  3. If required, repeat the clustering analysis. Prepare a `input_cluster.src` file and submit the  task as usual:
     ```
     qapta do_cluster apt_MD/input_cluster.src
     ```    
  5. For the best RMSD threshold, use ChimeraX to visualize the top 3 clusters
     ```
     chimerax rep.c0.pdb rep.c1.pdb rep.c2.pdb
     ```    
     Using the `match_maker` command in Chimerax superpose the models (for example, `match #2-3 to #1` )
  6. In the same Chimerax session, you can fetch the **experimental NMR structure** if you have the PDB id code. Superpose the PDB structure onto your cluster representatives. *Hint:* Hide most of the NMR models to better compare the various structures.
  7. Using ChimeraX, do not forget to save a high resolution `.png` file showing the best view of your models!
  8. Annotate the 3D structure of the top clusters using `x3dna-dssr`. This will give you much detailed information about the base pairing/stacking contacts. Obtain also the 2D Varna plots associated to the resulting secondary structure (convert `.dbn` using  `dbn2varna`). 

#### `do_mmpbsa.sh`

* Move to `6.ANALYSIS\MMPBSA\PDIE_1` and perform the following actions:
  1. Take note of the mean  value and uncertainty of  the MM/PBSA energy (see the `G_MMPBSA.med` file)
  2. How many MD frames were used in the MM/PBSA calculations? What is the time spacing between them?
  3. Examine the $G$ plot as a function of the number of snapshots. Is the system well *equilibrated*?
  4. Redo the statistics using the `$APTAMD/RUNMD/stat_plot.sh` script. Select only the last 50% of the data.  How does the mean $G$ change? 
  5. **Repeat the MM/PBSA calculations**, but including now  a reasonable number of $\ce{Na+}$ ions. From the information produced by `do_snapshots`,  you can find out how many ions are  at <3.0 $\rm{\mathring{A}}$ from solute atoms with a $\sim$90% probability.  You must carefully examine the possible options of the `do_mmpbsa.sh` script. Prepare an input_mmpbsa.src` script and submit the job:
  ```
  qapta do_mmpbsa apt_MD/input_mmpbsa.src
  ```
  6. After 10-20 min, the MM/PBSA should be finished. Examine the relevant MM/PBSA energies in the `6.ANALYSIS/MMPBSA_NA_X/PDIE_1`. What is the effect of including the  $\ce{Na+}$ ions? Are the $G$ energies from the two runs directly comparable?


#### `do_sconform.sh`

* Move to `6.ANALYSIS\SCONFORM` and perform the following actions:
   1. Visualize the convergence entropy plot. What is your assessment of the sampling convergence? If necessary, how could it be improved?
   2. What is the value of the best $S_{conform}$ estimation? How can we use it to refine the energetic scoring of the MD simulation?
   3. If you inspect the `cencalc_prep.out` file, you will find data concerning the rate of conformational change and individual entropies for all the relevant torsion angles. Which are the *fastest* and *slowest* torsional motions?

